In [1]:
!git clone https://github.com/spMohanty/PlantVillage-Dataset

Cloning into 'PlantVillage-Dataset'...
remote: Enumerating objects: 163264, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 163264 (delta 16), reused 25 (delta 9), pack-reused 163229 (from 1)
Receiving objects: 100% (163264/163264), 2.00 GiB | 33.41 MiB/s, done.
Resolving deltas: 100% (115/115), done.
Updating files: 100% (182404/182404), done.


In [2]:
cd PlantVillage-Dataset

/content/PlantVillage-Dataset


In [3]:
!pip install tensorflow_model_optimization

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 7.8 MB/s eta 0:00:00


In [4]:
# ================== IMPORTS ==================
import os, shutil, numpy as np, tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from tqdm import tqdm
import cv2, matplotlib.pyplot as plt

# Ensure compatibility for clustering API
# !pip install -q tensorflow-model-optimization==0.8.0
import tensorflow_model_optimization as tfmot


In [5]:

# ================== SE AND RESIDUAL BLOCKS ==================
def se_block(input_tensor, ratio=8):
    filters = int(input_tensor.shape[-1])
    se = layers.GlobalAveragePooling2D()(input_tensor)
    se = layers.Dense(filters // ratio, activation='relu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    se = layers.Reshape([1, 1, filters])(se)
    return layers.multiply([input_tensor, se])

def residual_dw_block(inputs, filters, strides=1):
    x = layers.DepthwiseConv2D(kernel_size=3, strides=strides, padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)
    x = layers.Conv2D(filters, kernel_size=1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = se_block(x)
    if strides == 1 and inputs.shape[-1] == filters:
        x = layers.Add()([inputs, x])
    return x

# ================== BUILD SE-RESNET MODEL ==================
def build_custom_se_resnet(input_shape=(64,64,3), num_classes=9):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(16, (3,3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)
    x = residual_dw_block(x, 32)
    x = residual_dw_block(x, 32)
    x = residual_dw_block(x, 64, strides=2)
    x = residual_dw_block(x, 64)
    x = residual_dw_block(x, 128, strides=2)
    x = residual_dw_block(x, 128)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(64, activation='swish', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = models.Model(inputs, outputs)
    return model


In [6]:

# ================== DATASET SETUP ==================
SOURCE_DIR = '/content/PlantVillage-Dataset/raw/segmented'
TARGET_BASE = '/content/PlantVillage-Dataset/processed_data'
IMG_SIZE = 64
BATCH_SIZE = 32

selected_classes = [
    "Cherry_(including_sour)___healthy",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Peach___Bacterial_spot",
    "Potato___Early_blight",
    "Potato___healthy",
    "Strawberry___Leaf_scorch",
    "Tomato___healthy",
    "Tomato___Leaf_Mold",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus"
]


In [7]:
import os, shutil
from sklearn.model_selection import train_test_split

if os.path.exists(TARGET_BASE):
    shutil.rmtree(TARGET_BASE)

os.makedirs(os.path.join(TARGET_BASE, 'train'), exist_ok=True)
os.makedirs(os.path.join(TARGET_BASE, 'val'), exist_ok=True)
os.makedirs(os.path.join(TARGET_BASE, 'test'), exist_ok=True)

for cls in selected_classes:
    cls_path = os.path.join(SOURCE_DIR, cls)

    if not os.path.exists(cls_path):
        print(f"Warning: {cls_path} missing — skipping class.")
        continue

    images = [
        f for f in os.listdir(cls_path)
        if os.path.isfile(os.path.join(cls_path, f))
        and f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
    ]

    if not images:
        print(f"Warning: No images found in {cls_path} — skipping class.")
        continue

    train_files, temp_files = train_test_split(
        images,
        test_size=0.30,
        random_state=42,
        shuffle=True
    )

    val_files, test_files = train_test_split(
        temp_files,
        test_size=0.50,
        random_state=42,
        shuffle=True
    )

    for group, files in [
        ('train', train_files),
        ('val', val_files),
        ('test', test_files)
    ]:
        out_dir = os.path.join(TARGET_BASE, group, cls)
        os.makedirs(out_dir, exist_ok=True)

        for f in files:
            shutil.copy(
                os.path.join(cls_path, f),
                os.path.join(out_dir, f)
            )

print("Dataset preparation complete.")

Dataset preparation complete.


In [8]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.3,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.7, 1.3]
)

val_test_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    os.path.join(TARGET_BASE, 'train'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_data = val_test_gen.flow_from_directory(
    os.path.join(TARGET_BASE, 'val'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_data = val_test_gen.flow_from_directory(
    os.path.join(TARGET_BASE, 'test'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=1,
    class_mode='categorical',
    shuffle=False
)

class_names = list(train_data.class_indices.keys())

print("Train samples:", train_data.samples)
print("Validation samples:", val_data.samples)
print("Test samples:", test_data.samples)
print("Classes:", class_names)

Found 10003 images belonging to 9 classes.
Found 2146 images belonging to 9 classes.
Found 2148 images belonging to 9 classes.
Train samples: 10003
Validation samples: 2146
Test samples: 2148
Classes: ['Cherry_(including_sour)___healthy', 'Corn_(maize)___Northern_Leaf_Blight', 'Peach___Bacterial_spot', 'Potato___Early_blight', 'Potato___healthy', 'Strawberry___Leaf_scorch', 'Tomato___Leaf_Mold', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___healthy']


In [9]:
base_model = build_custom_se_resnet(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    num_classes=len(class_names)
)

base_model.summary()

clustering_params = {
    'number_of_clusters': 8,
    'cluster_centroids_init': tfmot.clustering.keras.CentroidInitialization.KMEANS_PLUS_PLUS
}

try:
    clustered_model = tfmot.clustering.keras.cluster_weights(
        base_model,
        **clustering_params
    )
    print("Clustered model created successfully.")

except Exception as e:
    print("Clustering failed:", e)
    print("Using base model instead.")
    clustered_model = base_model

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 64, 64, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 64, 64,    │        448 │ input_layer[0][0] │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 64, 64,    │         64 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d    │ (None, 64, 64,    │        144 │ activation[0][0]  │
│ (DepthwiseConv2D)   │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │         64 │ depthwise_conv2d… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 64, 64,    │        512 │ activation_1[0][… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        128 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ batch_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 4)         │        132 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │        160 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 1, 32)  │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 64, 64,    │          0 │ batch_normalizat… │
│                     │ 32)               │            │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_1  │ (None, 64, 64,    │        288 │ multiply[0][0]    │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        128 │ depthwise_conv2d… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 64,    │      1,024 │ activation_2[0][

 Total params: 59,025 (230.57 KB)

 Trainable params: 57,425 (224.32 KB)

 Non-trainable params: 1,600 (6.25 KB)

Clustered model created successfully.


In [10]:
model_path = "/content/se_resnet_clustered_50epochs.keras"

try:
    clustered_model = tfmot.clustering.keras.cluster_weights(
        base_model,
        **clustering_params
    )

    if clustered_model is None:
        print("Clustering returned None. Using base_model instead.")
        clustered_model = base_model
    else:
        print("Clustered model created successfully.")

except Exception as e:
    print("Clustering failed:", e)
    print("Using base_model instead.")
    clustered_model = base_model

Clustering returned None. Using base_model instead.


In [11]:
clustered_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

In [12]:
print(type(clustered_model))

<class 'keras.src.models.functional.Functional'>


In [13]:
model_path = "/content/se_resnet_clustered_50epochs.keras"

if os.path.exists(model_path):
    print("Loading saved clustered model...")
    clustered_model = tf.keras.models.load_model(model_path, compile=False)
    clustered_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=['accuracy']
    )

else:
    print("Warning: 'clustered_model' was not found in global scope for training. Re-initializing...")

    clustered_model = tfmot.clustering.keras.cluster_weights(
        base_model,
        **clustering_params
    )

    if clustered_model is None:
        print("Warning: Clustering failed or returned None during re-initialization for training. Proceeding with the base model.")
        clustered_model = base_model

    clustered_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=['accuracy']
    )

    early_stopping = callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )

    lr_scheduler = callbacks.LearningRateScheduler(
        lambda epoch: 0.001 * 0.5 * (1 + np.cos(np.pi * epoch / 50))
    )

    checkpoint = callbacks.ModelCheckpoint(
        model_path,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )

    clustered_model.fit(
        train_data,
        epochs=50,
        validation_data=val_data,   
        callbacks=[early_stopping, lr_scheduler, checkpoint],
        verbose=1
    )

Epoch 1/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step - accuracy: 0.6107 - loss: 1.4466
Epoch 1: val_accuracy improved from None to 0.16076, saving model to /content/se_resnet_clustered_50epochs.keras

Epoch 1: finished saving model to /content/se_resnet_clustered_50epochs.keras
313/313 ━━━━━━━━━━━━━━━━━━━━ 97s 180ms/step - accuracy: 0.7272 - loss: 1.1718 - val_accuracy: 0.1608 - val_loss: 2.3043 - learning_rate: 0.0010
Epoch 2/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - accuracy: 0.8425 - loss: 0.8904
Epoch 2: val_accuracy improved from 0.16076 to 0.80336, saving model to /content/se_resnet_clustered_50epochs.keras

Epoch 2: finished saving model to /content/se_resnet_clustered_50epochs.keras
313/313 ━━━━━━━━━━━━━━━━━━━━ 29s 94ms/step - accuracy: 0.8576 - loss: 0.8577 - val_accuracy: 0.8034 - val_loss: 0.9353 - learning_rate: 9.9901e-04
Epoch 3/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.8935 - loss: 0.7811
Epoch 3: val_accuracy did not improve from 0.80336
313/313 ━

In [14]:
try:
    final_model = tfmot.clustering.keras.strip_clustering(clustered_model)
except Exception as e:
    print(f"Warning: Stripping clustering failed: {e}. Using clustered model as final model.")
    final_model = clustered_model
final_model.save("se_resnet_clustered_stripped.keras")

# ================== EVALUATE FLOAT32 MODEL ==================
y_true, y_pred = [], []
for i in tqdm(range(len(test_data))):
    img, label = test_data[i]
    pred = final_model.predict(img, verbose=0)[0]
    y_true.append(np.argmax(label[0]))
    y_pred.append(np.argmax(pred))

print("\nFloat32 Clustered-Stripped Model Accuracy:")
print(f"Top-1 Accuracy: {accuracy_score(y_true, y_pred)*100:.2f}%")
print(classification_report(y_true, y_pred, target_names=class_names))

100%|██████████| 2148/2148 [02:59<00:00, 11.96it/s]


Float32 Clustered-Stripped Model Accuracy:
Top-1 Accuracy: 98.60%
                                        precision    recall  f1-score   support

     Cherry_(including_sour)___healthy       0.99      0.91      0.95       129
   Corn_(maize)___Northern_Leaf_Blight       1.00      1.00      1.00       148
                Peach___Bacterial_spot       1.00      0.97      0.99       345
                 Potato___Early_blight       0.96      1.00      0.98       150
                      Potato___healthy       0.66      1.00      0.79        23
              Strawberry___Leaf_scorch       0.99      1.00      0.99       167
                    Tomato___Leaf_Mold       0.98      0.97      0.97       143
Tomato___Tomato_Yellow_Leaf_Curl_Virus       1.00      1.00      1.00       804
                      Tomato___healthy       0.99      0.99      0.99       239

                              accuracy                           0.99      2148
                             macro avg       0.95  

In [15]:

# ================== REPRESENTATIVE DATASET FOR PTQ ==================
def representative_dataset():
    rep_data = train_gen.flow_from_directory(
        os.path.join(TARGET_BASE, 'train'),
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=1,
        shuffle=True
    )
    for i in range(1000):  # Increased for better quantization
        img, _ = next(rep_data)
        yield [img.astype(np.float32)]


In [16]:

# ================== CONVERT TO INT8 TFLITE ==================
converter = tf.lite.TFLiteConverter.from_keras_model(final_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_model = converter.convert()

tflite_path = "se_resnet_clustered_int8_50epochs.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)
model_size_kb = os.path.getsize(tflite_path) / 1024
print(f"INT8-only clustered model saved as {tflite_path}")
print(f"Model size: {model_size_kb:.2f} KB")


Saved artifact at '/tmp/tmp_lflzly1'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64, 64, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 9), dtype=tf.float32, name=None)
Captures:
  134478884319696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134478906947920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134478885008400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134478885009360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134478885006288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134478885007824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134478885008784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134478885008208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134478885010320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134478885009936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134478885009744:

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


INT8-only clustered model saved as se_resnet_clustered_int8_50epochs.tflite
Model size: 118.48 KB


In [17]:

# ================== EVALUATE INT8 TFLITE ==================
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
input_scale, input_zero_point = input_details[0]['quantization']
output_scale, output_zero_point = output_details[0]['quantization']

y_true, y_pred = [], []
for i in tqdm(range(len(test_data))):
    img, label = test_data[i]
    img_int8 = (img / input_scale + input_zero_point).astype(np.int8)
    interpreter.set_tensor(input_details[0]['index'], img_int8)
    interpreter.invoke()
    pred_int8 = interpreter.get_tensor(output_details[0]['index'])[0]
    pred = (pred_int8.astype(np.float32) - output_zero_point) * output_scale
    pred = tf.nn.softmax(pred).numpy()
    y_true.append(np.argmax(label[0]))
    y_pred.append(np.argmax(pred))

print("\nINT8-only Clustered Model Accuracy:")
print(f"Top-1 Accuracy: {accuracy_score(y_true, y_pred)*100:.2f}%")
print(classification_report(y_true, y_pred, target_names=class_names))

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
100%|██████████| 2148/2148 [00:07<00:00, 289.65it/s]


INT8-only Clustered Model Accuracy:
Top-1 Accuracy: 98.09%
                                        precision    recall  f1-score   support

     Cherry_(including_sour)___healthy       1.00      0.88      0.93       129
   Corn_(maize)___Northern_Leaf_Blight       0.99      0.99      0.99       148
                Peach___Bacterial_spot       1.00      0.96      0.98       345
                 Potato___Early_blight       0.95      1.00      0.97       150
                      Potato___healthy       0.57      1.00      0.73        23
              Strawberry___Leaf_scorch       0.99      1.00      1.00       167
                    Tomato___Leaf_Mold       0.99      0.95      0.97       143
Tomato___Tomato_Yellow_Leaf_Curl_Virus       1.00      1.00      1.00       804
                      Tomato___healthy       0.96      0.99      0.98       239

                              accuracy                           0.98      2148
                             macro avg       0.94      0.9

In [18]:
import os
size_kb = os.path.getsize(tflite_path) / 1024
print(f"Model size: {size_kb:.2f} KB")

Model size: 118.48 KB
